In [1]:
from pathlib import Path
import json
import random

In [2]:
FULL_JSON = Path("../dataset/full.json")
OURSRC_JSON = Path("../dataset/oursrc.json")
OUTPUT_DIR = Path("../dataset")

TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
SEED = 42
REMOVE_DUPLICATES = True

In [3]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"{path} must contain a JSON array")
    return data


def record_signature(item: dict):
    return (
        (item.get("instruction") or "").strip(),
        (item.get("answer") or "").strip(),
        (item.get("problem") or "").strip(),
    )


def deduplicate(records):
    seen = set()
    unique = []
    for item in records:
        sig = record_signature(item)
        if sig in seen:
            continue
        seen.add(sig)
        unique.append(item)
    return unique


def split_records(records, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42):
    total_ratio = train_ratio + val_ratio + test_ratio
    if abs(total_ratio - 1.0) > 1e-9:
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    rng = random.Random(seed)
    shuffled = records[:]
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    # Keep all remaining items in test so no sample is lost.
    n_test = n - n_train - n_val

    train = shuffled[:n_train]
    val = shuffled[n_train : n_train + n_val]
    test = shuffled[n_train + n_val : n_train + n_val + n_test]

    return train, val, test


def save_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [5]:
# 1) Load
full_data = load_json(FULL_JSON)
oursrc_data = load_json(OURSRC_JSON)

# 2) Merge
merged = [dict(item) for item in full_data] + [dict(item) for item in oursrc_data]

# 3) Optional dedup
if REMOVE_DUPLICATES:
    merged = deduplicate(merged)

# 4) Split
train_data, val_data, test_data = split_records(
    merged,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
)

# 5) Save
train_path = OUTPUT_DIR / "train.json"
val_path = OUTPUT_DIR / "val.json"
test_path = OUTPUT_DIR / "test.json"

save_json(train_path, train_data)
save_json(val_path, val_data)
save_json(test_path, test_data)

print(f"Loaded full.json: {len(full_data)}")
print(f"Loaded oursrc.json: {len(oursrc_data)}")
print(f"Merged records: {len(merged)}")
print(f"Train: {len(train_data)} -> {train_path}")
print(f"Val:   {len(val_data)} -> {val_path}")
print(f"Test:  {len(test_data)} -> {test_path}")

Loaded full.json: 898
Loaded oursrc.json: 183
Merged records: 962
Train: 769 -> ..\dataset\train.json
Val:   96 -> ..\dataset\val.json
Test:  97 -> ..\dataset\test.json


In [9]:
import os
import json
from datasets import Dataset, DatasetDict, Features, Value
from huggingface_hub import login


REPO_ID = "xunnhi/geometry-dataset"
PRIVATE = True

HF_TOKEN = os.getenv("HF_TOKEN", "")

TRAIN_FILE = OUTPUT_DIR / "train.json"
VAL_FILE = OUTPUT_DIR / "val.json"
TEST_FILE = OUTPUT_DIR / "test.json"

for fp in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    if not fp.exists():
        raise FileNotFoundError(f"Khong tim thay file: {fp}. Hay chay Cell 4 truoc.")

if not HF_TOKEN:
    raise ValueError("Thieu HF_TOKEN. Hay tao token write tren Hugging Face va set bien moi truong.")

# Co dinh schema de tranh loi kieu du lieu khong dong nhat khi convert Arrow/Parquet
TARGET_COLUMNS = ["id", "image_dir", "instruction", "answer", "problem"]
FEATURES = Features({col: Value("string") for col in TARGET_COLUMNS})


def normalize_records_for_hf(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        records = json.load(f)

    normalized = []
    for row in records:
        item = {}
        for col in TARGET_COLUMNS:
            value = row.get(col, "") if isinstance(row, dict) else ""

            if value is None:
                value = ""
            elif not isinstance(value, str):
                # Neu value la list/dict/so -> doi sang chuoi de dong nhat schema
                value = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)

            item[col] = value

        normalized.append(item)

    return normalized


train_records = normalize_records_for_hf(TRAIN_FILE)
val_records = normalize_records_for_hf(VAL_FILE)
test_records = normalize_records_for_hf(TEST_FILE)

train_ds = Dataset.from_list(train_records, features=FEATURES)
val_ds = Dataset.from_list(val_records, features=FEATURES)
test_ds = Dataset.from_list(test_records, features=FEATURES)

dataset = DatasetDict(
    {
        "train": train_ds,
        "validation": val_ds,
        "test": test_ds,
    }
)

login(token=HF_TOKEN)
dataset.push_to_hub(REPO_ID, private=PRIVATE)

print(f"Dataset: https://huggingface.co/datasets/{REPO_ID}")
print(dataset)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/594 [00:00<?, ?B/s]

c:\Users\Xuan Nhi\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Xuan Nhi\.cache\huggingface\hub\datasets--xunnhi--geometry-dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Dataset: https://huggingface.co/datasets/xunnhi/geometry-dataset
DatasetDict({
    train: Dataset({
        features: ['id', 'image_dir', 'instruction', 'answer', 'problem'],
        num_rows: 769
    })
    validation: Dataset({
        features: ['id', 'image_dir', 'instruction', 'answer', 'problem'],
        num_rows: 96
    })
    test: Dataset({
        features: ['id', 'image_dir', 'instruction', 'answer', 'problem'],
        num_rows: 97
    })
})


In [10]:
# Kiem tra nhanh dataset sau khi upload len Hugging Face Hub
from datasets import load_dataset

hub_ds = load_dataset(REPO_ID)

print("Splits:", hub_ds)
print("Train size:", len(hub_ds["train"]))
print("Validation size:", len(hub_ds["validation"]))
print("Test size:", len(hub_ds["test"]))

print("\nTrain sample:")
print(hub_ds["train"][0])

print("\nValidation sample:")
print(hub_ds["validation"][0])

print("\nTest sample:")
print(hub_ds["test"][0])

README.md:   0%|          | 0.00/625 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/86.3k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/17.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/18.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/769 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/96 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/97 [00:00<?, ? examples/s]

Splits: DatasetDict({
    train: Dataset({
        features: ['id', 'image_dir', 'instruction', 'answer', 'problem'],
        num_rows: 769
    })
    validation: Dataset({
        features: ['id', 'image_dir', 'instruction', 'answer', 'problem'],
        num_rows: 96
    })
    test: Dataset({
        features: ['id', 'image_dir', 'instruction', 'answer', 'problem'],
        num_rows: 97
    })
})
Train size: 769
Validation size: 96
Test size: 97

Train sample:
{'id': '', 'image_dir': 'images/img_119748.png', 'instruction': 'Tam giác ABC, góc ABC = 90, góc BAC = 45, góc ACB = 45', 'answer': '(triangle (A B C) (right B))\n(angle-measure B A C 45)\n(angle-measure A B C 90)\n(angle-measure A C B 45)', 'problem': 'Cho tam giác ABC với góc ABC = 90°, góc BAC = 45° và góc ACB = 45°. Tính độ dài cạnh AB nếu biết rằng cạnh AC = 10.'}

Validation sample:
{'id': '', 'image_dir': 'images/img_104586.png', 'instruction': 'Tam giác ABC, góc ABC = 90, góc BAC = 60, góc ACB = 30, đường tròn ngoại tiế